# Phase 3: GIS Processing

This notebook prepares the GIS layers for downstream merge and spatial feature generation.

Scope:
- load property, road, and facilities layers
- repair invalid geometries
- standardize all layers to a projected CRS
- save processed GIS layers to `data/interim/`


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.data_loader import load_facility_layer, load_property_layer, load_road_layer
from backend.src.gis_processing import process_all_gis_layers, save_processed_gis_layers

configure_logging()
ensure_directories()
PROJECT_ROOT, settings


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'),
 Settings(project_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'), backend_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/backend'), raw_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data'), interim_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim'), processed_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/processed'), reports_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports'), models_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/models'), logs_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/logs'), backend_logs_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/logs/backend'), frontend_logs_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/logs/frontend'), transaction

In [2]:
property_gdf = load_property_layer()
roads_gdf = load_road_layer()
facilities_gdf = load_facility_layer()

property_gdf.shape, roads_gdf.shape, facilities_gdf.shape


2026-05-01 18:52:38,582 | INFO | backend.src.data_loader | Loading shapefile from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\ai_usecase_data240326.shp
2026-05-01 18:53:52,330 | INFO | backend.src.data_loader | Loading shapefile from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\ai_usecase_road240326.shp
2026-05-01 18:53:58,605 | INFO | backend.src.data_loader | Loading shapefile from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\ai_usecase_facilities240326.shp


((111497, 29), (10345, 10), (238, 21))

In [3]:
{
    'property_crs': str(property_gdf.crs),
    'roads_crs': str(roads_gdf.crs),
    'facilities_crs': str(facilities_gdf.crs),
    'property_invalid': int((~property_gdf.geometry.is_valid).sum()),
    'roads_invalid': int((~roads_gdf.geometry.is_valid).sum()),
    'facilities_invalid': int((~facilities_gdf.geometry.is_valid).sum()),
}


{'property_crs': 'EPSG:4326',
 'roads_crs': 'EPSG:32645',
 'facilities_crs': 'EPSG:4326',
 'property_invalid': 27,
 'roads_invalid': 0,
 'facilities_invalid': 0}

In [4]:
processed_property, processed_roads, processed_facilities, summary = process_all_gis_layers(
    property_gdf,
    roads_gdf,
    facilities_gdf,
)
summary


GISProcessingSummary(target_crs='EPSG:32645', property_layer=LayerProcessingSummary(layer_name='property_layer', input_row_count=111497, output_row_count=111497, input_crs='EPSG:4326', output_crs='EPSG:32645', invalid_geometry_count_before=27, invalid_geometry_count_after=0, null_geometry_count_after=0, geometry_types_after={'Polygon': 111255, 'MultiPolygon': 242}), road_layer=LayerProcessingSummary(layer_name='road_layer', input_row_count=10345, output_row_count=10345, input_crs='EPSG:32645', output_crs='EPSG:32645', invalid_geometry_count_before=0, invalid_geometry_count_after=0, null_geometry_count_after=0, geometry_types_after={'LineString': 10298, 'MultiLineString': 47}), facilities_layer=LayerProcessingSummary(layer_name='facilities_layer', input_row_count=238, output_row_count=238, input_crs='EPSG:4326', output_crs='EPSG:32645', invalid_geometry_count_before=0, invalid_geometry_count_after=0, null_geometry_count_after=0, geometry_types_after={'Point': 238}))

In [5]:
processed_property.shape, processed_roads.shape, processed_facilities.shape


((111497, 29), (10345, 10), (238, 21))

In [6]:
property_path, roads_path, facilities_path, summary_path = save_processed_gis_layers(
    processed_property,
    processed_roads,
    processed_facilities,
    summary,
    settings.interim_data_dir,
    settings.reports_dir,
)
property_path, roads_path, facilities_path, summary_path


2026-05-01 18:54:04,386 | INFO | backend.src.gis_processing | Saved processed property layer to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\interim\property_gis.parquet
2026-05-01 18:54:04,392 | INFO | backend.src.gis_processing | Saved processed road layer to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\interim\roads_gis.parquet
2026-05-01 18:54:04,397 | INFO | backend.src.gis_processing | Saved processed facilities layer to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\interim\facilities_gis.parquet
2026-05-01 18:54:04,403 | INFO | backend.src.gis_processing | Saved GIS processing summary to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\phase_3_gis_processing_summary.json


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim/property_gis.parquet'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim/roads_gis.parquet'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim/facilities_gis.parquet'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/phase_3_gis_processing_summary.json'))

In [7]:
json.loads(Path(summary_path).read_text())


{'target_crs': 'EPSG:32645',
 'property_layer': {'layer_name': 'property_layer',
  'input_row_count': 111497,
  'output_row_count': 111497,
  'input_crs': 'EPSG:4326',
  'output_crs': 'EPSG:32645',
  'invalid_geometry_count_before': 27,
  'invalid_geometry_count_after': 0,
  'null_geometry_count_after': 0,
  'geometry_types_after': {'Polygon': 111255, 'MultiPolygon': 242}},
 'road_layer': {'layer_name': 'road_layer',
  'input_row_count': 10345,
  'output_row_count': 10345,
  'input_crs': 'EPSG:32645',
  'output_crs': 'EPSG:32645',
  'invalid_geometry_count_before': 0,
  'invalid_geometry_count_after': 0,
  'null_geometry_count_after': 0,
  'geometry_types_after': {'LineString': 10298, 'MultiLineString': 47}},
 'facilities_layer': {'layer_name': 'facilities_layer',
  'input_row_count': 238,
  'output_row_count': 238,
  'input_crs': 'EPSG:4326',
  'output_crs': 'EPSG:32645',
  'invalid_geometry_count_before': 0,
  'invalid_geometry_count_after': 0,
  'null_geometry_count_after': 0,
  'ge